# JARVIS v6.0 — Google Colab Test Environment

**Note**: Colab is suitable for TESTING only, not production (sessions expire, no persistent daemons).
For production: use a VPS (Hetzner CX32 ~€15/month or Oracle Cloud Free Tier).

This notebook lets you test the core components interactively.

In [ ]:
# Install core dependencies
!pip install -q faster-whisper edge-tts sentence-transformers duckduckgo-search aiosqlite json-repair
!pip install -q fastapi uvicorn[standard] aiohttp python-multipart tenacity

# Install Ollama (free local LLM)
!curl -fsSL https://ollama.ai/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)
!ollama pull llama3.2:3b

In [ ]:
import sys, os
# Clone repo or mount Google Drive
# Option 1: Clone from git
# !git clone https://github.com/your/jarvis /content/jarvis
# sys.path.insert(0, '/content/jarvis/src')

# Option 2: Mount Drive
# from google.colab import drive
# drive.mount('/content/drive')
# sys.path.insert(0, '/content/drive/MyDrive/jarvis/src')

# Set home directory
os.environ['JARVIS_HOME'] = '/content/jarvis_data'
os.makedirs('/content/jarvis_data', exist_ok=True)

In [ ]:
# Test FREE local LLM (Ollama)
import asyncio
from jarvis.llm.client import simple_completion

response = await simple_completion('Τι ώρα είναι στην Αθήνα τώρα;', task_type='simple_qa')
print('LLM Response:', response)

In [ ]:
# Test FREE TTS (edge-tts)
from jarvis.voice.tts import synthesize_full
from IPython.display import Audio

audio_bytes = await synthesize_full('Γεια σου, εγώ είμαι ο JARVIS.')
Audio(audio_bytes, autoplay=True)

In [ ]:
# Test memory system (SQLite)
from jarvis.memory.database import supervised_db_writer, db_write
from jarvis.memory.store import save_memory, search_memories
import asyncio

# Start DB in background
asyncio.create_task(supervised_db_writer())
await asyncio.sleep(1)

# Save a memory
await save_memory('Ο χρήστης προτιμά σύντομες απαντήσεις.', memory_type='semantic', importance=0.8)

# Search memories
results = await search_memories('προτιμήσεις χρήστη')
for r in results:
    print(f"[{r['time_human']}] {r['content'][:200]}")

In [ ]:
# Test agent with tools
from jarvis.agents.base import BaseAgent
from jarvis.tools.registry import get_tools_for_set

agent = BaseAgent(agent_id='colab_test')
tool_defs, handlers = get_tools_for_set(['web_search', 'memory_search'])
for td in tool_defs:
    agent.register_tool(td['name'], td['description'], td['input_schema'], handlers[td['name']])

result = await agent.run_task('Ψάξε στο web για τα τελευταία νέα στην τεχνολογία AI')
print('Result:', result.output[:1000])
print('Score:', result.score)

In [ ]:
# Start JARVIS API server with public URL (ngrok)
!pip install -q pyngrok
from pyngrok import ngrok
import nest_asyncio
nest_asyncio.apply()

# Start server
import uvicorn
from jarvis.api.main import app

# Get public URL
public_url = ngrok.connect(8080)
print(f'JARVIS API: {public_url}')
print(f'Dashboard: {public_url}/dashboard')

await uvicorn.serve(app, host='0.0.0.0', port=8080)